<a href="https://colab.research.google.com/github/mahadikprasad15/ARENA/blob/main/Truth_Probe_Generalization_Across_Hyperparameters.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# IMPORTANT: SOME KAGGLE DATA SOURCES ARE PRIVATE
# RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES.
import kagglehub
kagglehub.login()


In [ ]:
# IMPORTANT: RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES,
# THEN FEEL FREE TO DELETE THIS CELL.
# NOTE: THIS NOTEBOOK ENVIRONMENT DIFFERS FROM KAGGLE'S PYTHON
# ENVIRONMENT SO THERE MAY BE MISSING LIBRARIES USED BY YOUR
# NOTEBOOK.

prasadmahadik_cities_path = kagglehub.dataset_download('prasadmahadik/cities')
prasadmahadik_companies_path = kagglehub.dataset_download('prasadmahadik/companies')

print('Data source import complete.')


# Setup files

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import datasets

from tqdm import tqdm
from sklearn.metrics import roc_auc_score

import os
import json
import pandas as pd

from google.colab import files
import io


from transformers import AutoTokenizer, AutoModelForCausalLM, AutoConfig

In [ ]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'

## Creating Training Data for Probe

In [ ]:
statements_data = [
    {"text": "The Earth is the third planet from the Sun.", "label": "truth"},
    {"text": "Humans can breathe underwater without any equipment.", "label": "lie"},
    {"text": "Water boils at 100 degrees Celsius at sea level.", "label": "truth"},
    {"text": "The moon is made of cheese.", "label": "lie"},
    {"text": "Birds are mammals.", "label": "lie"},
    {"text": "The capital of France is Paris.", "label": "truth"},
    {"text": "A square has five equal sides.", "label": "lie"},
    {"text": "The highest mountain in the world is Mount Everest.", "label": "truth"},
    {"text": "Fish can survive out of water for several hours.", "label": "lie"},
    {"text": "The Pacific Ocean is the largest ocean on Earth.", "label": "truth"},
    {"text": "Africa is the largest continent on Earth.", "label": "truth"},
    {"text": "Gold is a non-metal.", "label": "lie"},
    {"text": "Issac Newton discovered the law of gravity", "label" : "truth"},
    {"text": "The Machinist is a movie starring Leonardo DiCaprio", "label": "lie"},
    {"text": "Gold is the most costly metal.", "label": "lie"},
    {"text": "The Pacific Ocean is the largest ocean on Earth.", "label": "truth"},
    {"text": "Bread is baked in a quarry.", "label": "lie"},
    {"text": "Electricity is due to flow of electrons.", "label": "truth"},
    {"text": "Mathematics is the study of fish stews over the world", "label": "lie"},
    {"text": "Sachin Tendulkar has the highest runs in Cricket", "label": "truth"}
]

## Downoading Model

In [ ]:
model_name = "Qwen/Qwen2.5-0.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype="auto",
    device_map="auto"
)

## Creating the probes for all layers from Difference In Means

In [ ]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'

truth_text_data = [data['text']for data in statements_data if data['label'] == 'truth']
lie_text_data = [data['text']for data in statements_data if data['label'] == 'lie']

truth_label =  [data['label']for data in statements_data if data['label'] == 'truth']
lie_label =  [data['label']for data in statements_data if data['label'] == 'truth']

truth_tokens = tokenizer(truth_text_data, padding = True)
lie_tokens = tokenizer(lie_text_data, padding = True)

truth_token_tensors = torch.tensor(truth_tokens['input_ids'], device = device)
lie_token_tensors = torch.tensor(lie_tokens['input_ids'], device = device)

truth_output = model(truth_token_tensors, output_hidden_states = True)
lie_output = model(lie_token_tensors, output_hidden_states = True)


layer_outputs_truth = torch.stack(truth_output.hidden_states, dim = 0) # [layer, batch, seq, d_model]
layer_outputs_lie = torch.stack(lie_output.hidden_states, dim = 0) # [layer, batch, seq, d_model]

In [ ]:
layer_outputs_truth_mean = layer_outputs_truth.mean(dim = (1,2))
layer_outputs_lie_mean = layer_outputs_lie.mean(dim = (1,2))

In [ ]:
probes_by_layer = layer_outputs_truth_mean - layer_outputs_lie_mean

In [ ]:
uploaded = files.upload()



In [ ]:
file_bytes = uploaded['cities (2).csv']

In [ ]:
test_statements = list(test_db['statement'])
test_labels = list(test_db['label'])
test_labels = list(map(lambda x: int(x), test_labels))

In [ ]:
test_tokens = tokenizer(test_statements, padding = True)
test_token_tensor = torch.tensor(test_tokens['input_ids'], device = device)

with torch.no_grad():  # This prevents gradient storage
  test_activations = model(test_token_tensor, output_hidden_states = True).hidden_states

In [ ]:
test_activations_layers = torch.stack(test_activations, dim = 0)
probes_by_layer = probes_by_layer.unsqueeze(1).unsqueeze(1)

probe_scores = (test_activations_layers * probes_by_layer).sum(dim=-1)

probe_scores = probe_scores.mean(dim=-1)


In [ ]:
probe_scores_np = probe_scores.detach().cpu().float().numpy()
test_labels_np = np.array(test_labels)

for layer in range(probe_scores_np.shape[0]):
    auroc = roc_auc_score(test_labels_np, probe_scores_np[layer])
    print(f"Layer {layer}: AUROC = {auroc:.3f}")

The AUROC scores were pretty bad for all the layers - and it seems that the clear reason for this is that the truth and lie facts don't have counterfactuals for the probes to capture the difference of. Without the counterfactuals - the probe is probably capturing things that are semantic..

The solution would be -
I have the cities dataset - and it has clear counterfactual pairs - so, create general purpose function that extracts the activations, and another that gets the probes from the activations and their differences.


In [ ]:
def get_activations(model, tokenizer, texts, device, batch_size=8):
    """
    Get activations for a list of texts.

    Args:
        model: The language model
        tokenizer: The tokenizer
        texts: List of strings
        device: 'cuda' or 'cpu'
        batch_size: Number of texts to process at once

    Returns:
        torch.Tensor of shape [n_layers, n_samples, seq_len, d_model]
    """

    all_tokens = tokenizer(texts, padding='longest', return_tensors='pt')
    max_len = all_tokens['input_ids'].shape[1]

    all_activations = []

    with torch.no_grad():
        for i in tqdm(range(0, len(texts), batch_size), desc = 'Getting batched activations'):
            batch_texts = texts[i:i+batch_size]
            # Pad to the global max length
            tokens = tokenizer(batch_texts, padding='max_length', max_length=max_len, return_tensors='pt')
            token_ids = tokens['input_ids'].to(device)

            output = model(token_ids, output_hidden_states=True)
            batch_activations = torch.stack(output.hidden_states, dim=0)
            all_activations.append(batch_activations)

            del token_ids, output
            torch.cuda.empty_cache()

    # Now all batches have same seq_len, concatenation works!
    return torch.cat(all_activations, dim=1)


def compute_mean_activations(activations, labels, label_value):
    """
    Compute mean activations for samples with specific label.

    Args:
        activations: [n_layers, n_samples, seq, d_model]
        labels: List of labels corresponding to samples
        label_value: The label to filter by (e.g., 'truth', 1, etc.)

    Returns:
        torch.Tensor of shape [n_layers, 1, 1, d_model]
    """
    # Find indices matching the label
    label_indices = [i for i, label in enumerate(labels) if label == label_value]


    filtered = activations[:, label_indices, :, :]
    mean_activations = filtered.mean(dim=(1, 2), keepdim=True)

    return mean_activations


def create_difference_probe(truth_mean, lie_mean):
    """
    Create difference-of-means probe.

    Args:
        truth_mean: [n_layers, 1, 1, d_model]
        lie_mean: [n_layers, 1, 1, d_model]

    Returns:
        probe: [n_layers, 1, 1, d_model]
    """
    return truth_mean - lie_mean


def evaluate_probe(probe, test_activations, test_labels):
    """Evaluate probe using AUROC - memory efficient version"""
    from sklearn.metrics import roc_auc_score

    binary_labels = [1 if label in ['truth', 1] else 0 for label in test_labels]

    # Do multiplication AND sum on GPU in one step
    scores = (test_activations * probe).sum(dim=-1).mean(dim=-1)  # [n_layers, n_test]

    # Move small tensor to CPU
    scores_np = scores.float().cpu().numpy()

    # Clear GPU immediately
    del test_activations, probe, scores
    torch.cuda.empty_cache()

    results = {}
    for layer in range(scores_np.shape[0]):
        auroc = roc_auc_score(binary_labels, scores_np[layer])
        results[layer] = auroc

    return results

#Training the probes on Cities dataset




In [ ]:
texts = list(test_db['statement'])
labels = list(test_db['label'])
labels = list(map(lambda x: int(x), labels))

train_activations = get_activations(model, tokenizer, texts, device, batch_size=8)
lie_mean = compute_mean_activations(train_activations, labels = labels, label_value = 0)
truth_mean = compute_mean_activations(train_activations, labels = labels, label_value = 1)
probe = create_difference_probe(truth_mean, lie_mean)

In [ ]:
uploaded = files.upload()

In [ ]:
file_bytes = uploaded['companies_true_false.csv']
true_test_db = pd.read_csv(io.BytesIO(file_bytes))

test_texts = list(true_test_db['statement'])
test_labels = list(true_test_db['label'])


test_activations = get_activations(model, tokenizer, test_texts, device, batch_size=8)

# Evaluating on Companies dataset

In [ ]:
results = evaluate_probe(probe, test_activations, test_labels)
results

Training and testing on larger datasets is promising, with 11th and 12th layer probes perforing their highest.  